In [2]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx

from pyproj import Transformer
from sklearn.neighbors import BallTree
from pathlib import Path
import requests
import zipfile

In [3]:
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

url_ubicacion_dataset = "https://datos.madrid.es/dataset/202468-0-intensidad-trafico"

response = requests.get(url_ubicacion_dataset)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

enlaces = []

for a in soup.find_all("a", href=True):
    href = urljoin(url_ubicacion_dataset, a["href"])
    texto = a.get_text(" ", strip=True)

    if "/download/" in href or href.lower().endswith((".csv", ".xlsx", ".zip")):
        enlaces.append({
            "texto": texto,
            "url": href
        })

df_enlaces = pd.DataFrame(enlaces).drop_duplicates()
df_enlaces.head(20)

,texto,url
0,Descarga,https://datos.madrid.es/dataset/202468-0-inten...
1,Descarga,https://datos.madrid.es/dataset/202468-0-inten...
2,Descarga,https://datos.madrid.es/dataset/202468-0-inten...
3,Descarga,https://datos.madrid.es/dataset/202468-0-inten...


In [7]:
df_enlaces[df_enlaces["url"].str.contains(".csv", case=False, na=False)]

,texto,url


In [9]:
url_csv = "https://datos.madrid.es/dataset/202468-0-intensidad-trafico/resource/202468-294-intensidad-trafico/download/202468-294-intensidad-trafico.csv"

df_medidores = pd.read_csv(
    url_csv,
    sep=";",
    encoding="latin1"
)

df_medidores.head()

,tipo_elem,distrito,id,cod_cent,nombre,utm_x,utm_y,longitud,latitud
0,other,1.0,6835,18RA28PM01,18RA28PM01,438764.313318,4.474327e+06,-3.721794,40.417315
1,other,9.0,1012,18RA66PM01,18RA66PM01,438740.943152,4.474610e+06,-3.722097,40.419861
2,URB,10.0,5035,95013,FRUELA N-S,438004.401612,4.473859e+06,-3.730705,40.413040
3,URB,5.0,5579,61068,Potosi E-O - Bolivia-Víctor Andrés Belaunde,442420.642251,4.478696e+06,-3.679095,40.456932
4,URB,5.0,5580,61069,Víctor Andrés Belaunde N-S - Cochabamba-Potosi,442366.670466,4.478601e+06,-3.679723,40.456073


In [10]:
df_medidores.columns

Index(['tipo_elem', 'distrito', 'id', 'cod_cent', 'nombre', 'utm_x', 'utm_y',
       'longitud', 'latitud'],
      dtype='object')

In [12]:
medidores = df_medidores[["id", "nombre", "utm_x", "utm_y", "longitud", "latitud"]].copy()

medidores = medidores.dropna(subset=["longitud", "latitud"])
medidores = medidores.drop_duplicates(subset=["id"])

medidores.head()

,id,nombre,utm_x,utm_y,longitud,latitud
0,6835,18RA28PM01,438764.313318,4.474327e+06,-3.721794,40.417315
1,1012,18RA66PM01,438740.943152,4.474610e+06,-3.722097,40.419861
2,5035,FRUELA N-S,438004.401612,4.473859e+06,-3.730705,40.413040
3,5579,Potosi E-O - Bolivia-Víctor Andrés Belaunde,442420.642251,4.478696e+06,-3.679095,40.456932
4,5580,Víctor Andrés Belaunde N-S - Cochabamba-Potosi,442366.670466,4.478601e+06,-3.679723,40.456073


In [13]:
print("Número de medidores:", len(medidores))
medidores.head()

Número de medidores: 5072


,id,nombre,utm_x,utm_y,longitud,latitud
0,6835,18RA28PM01,438764.313318,4.474327e+06,-3.721794,40.417315
1,1012,18RA66PM01,438740.943152,4.474610e+06,-3.722097,40.419861
2,5035,FRUELA N-S,438004.401612,4.473859e+06,-3.730705,40.413040
3,5579,Potosi E-O - Bolivia-Víctor Andrés Belaunde,442420.642251,4.478696e+06,-3.679095,40.456932
4,5580,Víctor Andrés Belaunde N-S - Cochabamba-Potosi,442366.670466,4.478601e+06,-3.679723,40.456073


In [14]:
medidores.dtypes

id            int64
nombre       object
utm_x       float64
utm_y       float64
longitud    float64
latitud     float64
dtype: object

In [15]:
medidores[["latitud", "longitud"]].describe()

,latitud,longitud
count,5072.000000,5072.000000
mean,40.430447,-3.684001
std,0.039163,0.042728
min,40.332454,-3.836886
25%,40.398978,-3.712553
50%,40.431302,-3.686923
75%,40.460080,-3.656194
max,40.515611,-3.551623


In [16]:
G_osm = ox.graph_from_place(
    "Madrid, Spain",
    network_type="drive",
    simplify=True
)

print("Nodos OSM:", len(G_osm.nodes))
print("Aristas OSM:", len(G_osm.edges))

Nodos OSM: 31453
Aristas OSM: 61857
